# Animal Kingdom species and suitability review

This notebook authenticates to the private bucket, streams the action-video archive once, extracts only the deterministic review sample, and saves labels back to the bucket. It does not use a GPU or train a model.

Run each cell in order. The extraction cell reads the complete compressed 15.6 GB archive but stores only approximately 203 selected videos on the temporary Colab disk. Do not disconnect while it runs.

In [ ]:
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'cat-behaviour-research-raluca-2026'
REPO_URL = 'https://github.com/ralucabiras/explainable-cat-behaviour-interpreter.git'
REPO_DIR = '/content/explainable-cat-behaviour-interpreter'
PLAN_PATH = f'{REPO_DIR}/backend/video_dataset/review/animal-kingdom-review-v1.plan.json'
LABELS_URI = f'gs://{BUCKET}/curated/review-v1/review-labels.json'

!gcloud config set project {PROJECT_ID}
!test -d {REPO_DIR}/.git || git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} pull --ff-only
print('Authentication and repository setup complete.')

In [ ]:
import json
import subprocess
from pathlib import Path

plan = json.loads(Path(PLAN_PATH).read_text())
extract_root = Path('/content/animal-kingdom-review')
extract_root.mkdir(parents=True, exist_ok=True)
members_path = extract_root / 'members.txt'
members = sorted({item['archive_member'] for item in plan['items']})
members_path.write_text('\n'.join(members) + '\n')
print(f"Plan: {plan['plan_version']}")
print(f'Selected source videos: {len(members)}')
print(f"Archive: {plan['source_archive_uri']}")

In [ ]:
def extract_review_videos():
    missing_before = [
        member for member in members if not (extract_root / member).is_file()
    ]
    if not missing_before:
        print(f'All {len(members)} review videos are already extracted.')
        return
    # Stream the archive once; never save the complete archive to Colab.
    gcloud = subprocess.Popen(
        ['gcloud', 'storage', 'cat', plan['source_archive_uri']],
        stdout=subprocess.PIPE,
    )
    tar = subprocess.run(
        ['tar', '-xzf', '-', '-C', str(extract_root), '-T', str(members_path)],
        stdin=gcloud.stdout,
    )
    if gcloud.stdout:
        gcloud.stdout.close()
    gcloud_code = gcloud.wait()
    if tar.returncode or gcloud_code:
        raise RuntimeError(
            f'Extraction failed: gcloud={gcloud_code}, tar={tar.returncode}'
        )
    missing = [
        member for member in members if not (extract_root / member).is_file()
    ]
    if missing:
        raise RuntimeError(
            f'{len(missing)} planned archive members were missing; first: {missing[:5]}'
        )
    print(f'Extracted and verified {len(members)} review videos.')

extract_review_videos()

## Review labels

For each video, select the visible species and whether the clip is suitable for learning the listed observable actions. Use `unclear` instead of guessing. A checkpoint is uploaded every five saved reviews and when **Checkpoint now** is pressed.

In [ ]:
import sys

sys.path.insert(0, f'{REPO_DIR}/backend')
from app.video_dataset.colab_review import launch_review  # noqa: E402

controller = launch_review(plan, extract_root, LABELS_URI)
labels = controller.labels
checkpoint = controller.checkpoint

In [ ]:
from collections import Counter

# Run after finishing or whenever you want a progress summary.
checkpoint()
print('Reviewed:', len(labels), '/', len(plan['items']))
print('Species:', dict(Counter(item['species'] for item in labels.values())))
print('Suitability:', dict(Counter(item['suitability'] for item in labels.values())))
print('Saved privately to:', LABELS_URI)